# Part 10: Anomaly Detection

[← Back to Index](Index.ipynb)

**Quick Reference Guide for Detecting Outliers and Anomalies**

---
## 10.1 Statistical Methods

**Concept:** Use statistical properties to identify unusual data points

**When to use:** Simple datasets, interpretable results needed

### Z-Score Method

**Concept:** Points beyond ±3 standard deviations are anomalies

**Assumption:** Data is normally distributed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Generate sample data with anomalies
np.random.seed(42)
normal_data = np.random.randn(1000) * 10 + 50
anomalies = np.random.uniform(100, 120, 20)
data = np.concatenate([normal_data, anomalies])
np.random.shuffle(data)

# Z-score method
def detect_anomalies_zscore(data, threshold=3):
    """Detect anomalies using z-score"""
    z_scores = np.abs(stats.zscore(data))
    anomalies = z_scores > threshold
    return anomalies, z_scores

anomalies_mask, z_scores = detect_anomalies_zscore(data)
print(f"Number of anomalies detected: {anomalies_mask.sum()}")

# Visualize
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(range(len(data)), data, c=anomalies_mask, cmap='RdYlGn_r', alpha=0.6)
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Z-Score Anomaly Detection')
plt.colorbar(label='Anomaly')

plt.subplot(1, 2, 2)
plt.hist(data, bins=50, alpha=0.7, label='Normal')
plt.hist(data[anomalies_mask], bins=20, alpha=0.7, label='Anomalies', color='red')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution')
plt.legend()

plt.tight_layout()
plt.show()

### IQR (Interquartile Range) Method

**Concept:** Points outside Q1-1.5IQR or Q3+1.5IQR are anomalies

**Advantage:** Robust to outliers, no normality assumption

In [ ]:
def detect_anomalies_iqr(data, multiplier=1.5):
    """Detect anomalies using IQR method"""
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    anomalies = (data < lower_bound) | (data > upper_bound)
    
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    
    return anomalies, lower_bound, upper_bound

anomalies_iqr, lower, upper = detect_anomalies_iqr(data)
print(f"\nNumber of anomalies: {anomalies_iqr.sum()}")

# Box plot
plt.figure(figsize=(10, 6))
plt.boxplot(data, vert=False)
plt.scatter(data[anomalies_iqr], [1]*anomalies_iqr.sum(), 
           color='red', s=100, alpha=0.5, label='Anomalies')
plt.xlabel('Value')
plt.title('IQR Method - Box Plot')
plt.legend()
plt.grid(True)
plt.show()

### Modified Z-Score (Using Median)

**Concept:** More robust version using median and MAD

**Formula:** Modified Z = 0.6745 * (x - median) / MAD

In [ ]:
def detect_anomalies_modified_zscore(data, threshold=3.5):
    """Detect anomalies using modified z-score"""
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    
    modified_z_scores = 0.6745 * (data - median) / mad
    anomalies = np.abs(modified_z_scores) > threshold
    
    return anomalies, modified_z_scores

anomalies_mod, mod_z = detect_anomalies_modified_zscore(data)
print(f"Anomalies detected: {anomalies_mod.sum()}")

---
## 10.2 Machine Learning Methods

### Isolation Forest

**Concept:** Anomalies are easier to isolate (fewer splits needed)

**Advantages:**
- Works well with high-dimensional data
- Fast and scalable
- No assumptions about data distribution

**When to use:** Large datasets, many features

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_blobs

# Generate 2D data for visualization
X, _ = make_blobs(n_samples=300, centers=1, cluster_std=1.0, random_state=42)
# Add anomalies
X_anomalies = np.random.uniform(low=-8, high=8, size=(20, 2))
X = np.vstack([X, X_anomalies])

# Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,      # number of trees
    contamination=0.1,     # expected proportion of anomalies
    max_samples='auto',    # samples to draw for each tree
    random_state=42
)

# Fit and predict (-1 for anomalies, 1 for normal)
predictions = iso_forest.fit_predict(X)
anomaly_scores = iso_forest.score_samples(X)  # anomaly scores

anomalies = predictions == -1
print(f"Anomalies detected: {anomalies.sum()}")

# Visualize
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(X[~anomalies, 0], X[~anomalies, 1], c='blue', label='Normal', alpha=0.6)
plt.scatter(X[anomalies, 0], X[anomalies, 1], c='red', label='Anomaly', alpha=0.8, s=100)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Isolation Forest - Anomaly Detection')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(range(len(anomaly_scores)), anomaly_scores, c=predictions, cmap='RdYlGn')
plt.xlabel('Sample Index')
plt.ylabel('Anomaly Score')
plt.title('Anomaly Scores (lower = more anomalous)')
plt.colorbar(label='Prediction')
plt.grid(True)

plt.tight_layout()
plt.show()

### One-Class SVM

**Concept:** Learn boundary around normal data

**When to use:** Clear dense cluster of normal data

In [ ]:
from sklearn.svm import OneClassSVM

# One-Class SVM
ocsvm = OneClassSVM(
    kernel='rbf',          # 'rbf', 'linear', 'poly'
    gamma='auto',          # kernel coefficient
    nu=0.1                 # upper bound on fraction of outliers
)

predictions = ocsvm.fit_predict(X)
anomalies_svm = predictions == -1

print(f"Anomalies detected: {anomalies_svm.sum()}")

# Visualize decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 100),
                     np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 100))
Z = ocsvm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap='RdYlGn_r', alpha=0.3)
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
plt.scatter(X[~anomalies_svm, 0], X[~anomalies_svm, 1], c='blue', label='Normal', alpha=0.6)
plt.scatter(X[anomalies_svm, 0], X[anomalies_svm, 1], c='red', label='Anomaly', s=100, alpha=0.8)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('One-Class SVM')
plt.legend()
plt.grid(True)
plt.show()

### Local Outlier Factor (LOF)

**Concept:** Compare local density to neighbors' density

**Advantages:**
- Detects local anomalies
- Works with varying densities

**When to use:** Data with multiple clusters of different densities

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# LOF
lof = LocalOutlierFactor(
    n_neighbors=20,        # number of neighbors
    contamination=0.1,     # expected proportion of outliers
    novelty=False          # False for fit_predict, True for predict on new data
)

predictions = lof.fit_predict(X)
anomaly_scores = lof.negative_outlier_factor_  # lower = more anomalous

anomalies_lof = predictions == -1
print(f"Anomalies detected: {anomalies_lof.sum()}")

# Visualize
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(X[~anomalies_lof, 0], X[~anomalies_lof, 1], 
           c='blue', label='Normal', alpha=0.6)
plt.scatter(X[anomalies_lof, 0], X[anomalies_lof, 1], 
           c='red', label='Anomaly', s=100, alpha=0.8)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Local Outlier Factor (LOF)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(range(len(anomaly_scores)), anomaly_scores, c=predictions, cmap='RdYlGn')
plt.xlabel('Sample Index')
plt.ylabel('LOF Score (lower = more anomalous)')
plt.title('LOF Scores')
plt.colorbar(label='Prediction')
plt.grid(True)

plt.tight_layout()
plt.show()

### Autoencoders for Anomaly Detection

**Concept:** Train to reconstruct normal data; anomalies have high reconstruction error

**When to use:** Complex high-dimensional data, deep learning appropriate

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

# Generate high-dimensional data
np.random.seed(42)
n_samples = 1000
n_features = 20

X_normal = np.random.randn(n_samples, n_features)
X_anomalies = np.random.uniform(-4, 4, (50, n_features))
X_full = np.vstack([X_normal, X_anomalies])

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

# Split (use only normal data for training)
X_train = X_scaled[:n_samples]
X_test = X_scaled

# Build Autoencoder
encoding_dim = 5

encoder = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(10, activation='relu'),
    layers.Dense(encoding_dim, activation='relu')
])

decoder = keras.Sequential([
    layers.Input(shape=(encoding_dim,)),
    layers.Dense(10, activation='relu'),
    layers.Dense(n_features, activation='linear')
])

autoencoder = keras.Sequential([encoder, decoder])
autoencoder.compile(optimizer='adam', loss='mse')

# Train on normal data only
history = autoencoder.fit(
    X_train, X_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Reconstruct and calculate error
X_reconstructed = autoencoder.predict(X_test)
reconstruction_errors = np.mean(np.square(X_test - X_reconstructed), axis=1)

# Set threshold (e.g., 95th percentile of training errors)
threshold = np.percentile(reconstruction_errors[:n_samples], 95)
anomalies_ae = reconstruction_errors > threshold

print(f"Anomalies detected: {anomalies_ae.sum()}")
print(f"Threshold: {threshold:.4f}")

# Visualize
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Autoencoder Training')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(range(len(reconstruction_errors)), reconstruction_errors, 
           c=anomalies_ae, cmap='RdYlGn_r', alpha=0.6)
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold')
plt.xlabel('Sample Index')
plt.ylabel('Reconstruction Error')
plt.title('Autoencoder Anomaly Detection')
plt.colorbar(label='Anomaly')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

---
## 10.3 Advanced Techniques

### Time Series Anomaly Detection

**Methods:**
1. Statistical (moving average, STL decomposition)
2. ARIMA-based residuals
3. LSTM Autoencoder

In [ ]:
# Generate time series with anomalies
np.random.seed(42)
n = 1000
time = np.arange(n)
trend = 0.01 * time
seasonal = 5 * np.sin(2 * np.pi * time / 50)
noise = np.random.randn(n)
ts = trend + seasonal + noise

# Add anomalies
anomaly_indices = np.random.choice(n, 20, replace=False)
ts[anomaly_indices] += np.random.uniform(10, 20, 20)

# Method 1: Moving average + std
window = 20
rolling_mean = pd.Series(ts).rolling(window=window).mean()
rolling_std = pd.Series(ts).rolling(window=window).std()

upper_bound = rolling_mean + 3 * rolling_std
lower_bound = rolling_mean - 3 * rolling_std

anomalies_ts = (ts > upper_bound) | (ts < lower_bound)

# Visualize
plt.figure(figsize=(14, 6))
plt.plot(time, ts, label='Time Series', alpha=0.7)
plt.plot(time, rolling_mean, label='Rolling Mean', color='blue')
plt.fill_between(time, lower_bound, upper_bound, alpha=0.2, label='Normal Range')
plt.scatter(time[anomalies_ts], ts[anomalies_ts], 
           color='red', s=100, label='Anomalies', zorder=5)
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Time Series Anomaly Detection')
plt.legend()
plt.grid(True)
plt.show()

print(f"Anomalies detected: {anomalies_ts.sum()}")

### Multivariate Anomaly Detection

**Challenge:** Detect anomalies in multiple correlated features

In [ ]:
from sklearn.covariance import EllipticEnvelope

# Generate multivariate data
np.random.seed(42)
X_normal = np.random.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], 300)
X_anomalies = np.random.uniform(-5, 5, (20, 2))
X_multi = np.vstack([X_normal, X_anomalies])

# Elliptic Envelope (assumes Gaussian distribution)
envelope = EllipticEnvelope(contamination=0.1, random_state=42)
predictions = envelope.fit_predict(X_multi)
anomalies_multi = predictions == -1

print(f"Anomalies detected: {anomalies_multi.sum()}")

# Visualize
plt.figure(figsize=(10, 8))
plt.scatter(X_multi[~anomalies_multi, 0], X_multi[~anomalies_multi, 1], 
           c='blue', label='Normal', alpha=0.6)
plt.scatter(X_multi[anomalies_multi, 0], X_multi[anomalies_multi, 1], 
           c='red', label='Anomaly', s=100, alpha=0.8)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Multivariate Anomaly Detection (Elliptic Envelope)')
plt.legend()
plt.grid(True)
plt.show()

---
### Method Comparison

| Method | Type | Supervised | Scalability | Best Use Case |
|--------|------|------------|-------------|---------------|
| Z-Score | Statistical | No | Fast | Univariate, normal distribution |
| IQR | Statistical | No | Fast | Univariate, robust to outliers |
| Isolation Forest | ML | No | Fast | High-dimensional, large data |
| One-Class SVM | ML | No | Medium | Clear normal cluster |
| LOF | ML | No | Slow | Varying densities |
| Autoencoder | DL | No | Medium | Complex patterns, large data |
| LSTM | DL | No | Medium | Time series, sequential |
| Elliptic Envelope | Statistical | No | Fast | Multivariate Gaussian |

---
### Evaluation Metrics

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Assuming we have true labels
y_true = np.array([0]*300 + [1]*20)  # 0=normal, 1=anomaly
y_pred = (predictions == -1).astype(int)

# Metrics
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Visualize
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

---
### Quick Reference Guide

**Method Selection:**

1. **Simple univariate data:**
   - Use Z-Score or IQR
   
2. **Multivariate tabular data:**
   - Small: LOF, One-Class SVM
   - Large: Isolation Forest
   
3. **Time series:**
   - Statistical: Moving average + std
   - ML: LSTM Autoencoder
   
4. **High-dimensional:**
   - Isolation Forest
   - Autoencoder
   
5. **Real-time:**
   - Online algorithms
   - Sliding window approach

**Best Practices:**
- Understand domain (what is "normal"?)
- Set contamination parameter carefully
- Use ensemble of methods
- Validate with domain experts
- Monitor false positives vs false negatives
- Consider cost of missing anomaly vs false alarm
- Use appropriate evaluation metrics
- Retrain periodically as "normal" changes

**Common Applications:**
- Fraud detection
- Network intrusion detection
- Manufacturing defect detection
- Healthcare monitoring
- System/server monitoring
- Predictive maintenance

---
[← Back to Index](Index.ipynb)